# Exploratory Data Analysis

This notebook contains EDA of life expectancy and HALE data from WHO, and exploration of mortality indicators from WHO and IHME.
See the data inventory for more information about available indicators.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from utils import (
    decorate, underride, configure_plot_style, AIBM_COLORS, 
    code_to_who_country, code_to_wef_country, write_html_table,
    load_and_inventory, compute_gender_gap, summarize_gap, scatter_plot,
    get_oecd, summarize_years, plot_cdfs, plot_distributions,
    column_name_mapping, oecd_codes
)

configure_plot_style()

# Set cutoff year for temporal analysis (excludes 2020+ to avoid COVID-19 distortions)
cutoff_year = 2019

## WHO HALE data

**Healthy Life Expectancy (HALE) at birth** - The average number of years that a person can expect to live in "full health" by taking into account years lived in less than full health due to disease and/or injury. This is the **target variable** for the analysis. The gender gap (Male HALE - Female HALE) measures the difference in healthy life expectancy between men and women.

**Indicator Code**: WHOSIS_000002  
**Relevance**: Direct measure of the outcome we're trying to explain. Gender differences in HALE reflect the cumulative impact of all mortality and morbidity factors that differentially affect men and women.

Downloaded using the GHO OData API (who_data.py)

https://www.who.int/data/gho/info/gho-odata-api

In [ ]:
filename = '../data/who_hale_data.csv'
hale, years = load_and_inventory(filename)

In [ ]:
d = {'SEX_BTSX': 'Both', 'SEX_FMLE': 'Female', 'SEX_MLE': 'Male', }
hale['Sex'] = hale['Sex'].replace(d)

In [ ]:
hale.head()

In [ ]:
col = 'HALE_Years'
hale_gap = summarize_gap(hale, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(hale_gap)

## WHO Life Expectancy data

**Life Expectancy at birth** - The average number of years that a person can expect to live, regardless of health status. This is the **secondary target variable** for the analysis, allowing comparison of which factors explain the gender gap in overall life expectancy versus healthy life expectancy. The gender gap (Female LE - Male LE) measures the difference in life expectancy between women and men.

**Indicator Code**: WHOSIS_000001  
**Relevance**: Life expectancy captures all years lived (healthy and unhealthy), while HALE focuses on healthy years only. Both are calculated from birth, so both should be affected by the same mortality patterns. The relative importance of early-life vs adult mortality may differ between the two outcomes.

Downloaded using the GHO OData API (who_data.py)

https://www.who.int/data/gho/info/gho-odata-api

In [ ]:
filename = '../data/who_life_expectancy_data.csv'
le, years = load_and_inventory(filename)

In [ ]:
d = {'SEX_BTSX': 'Both', 'SEX_FMLE': 'Female', 'SEX_MLE': 'Male', }
le['Sex'] = le['Sex'].replace(d)

In [ ]:
le.head()

In [ ]:
col = 'LifeExpectancy_Years'
le_gap = summarize_gap(le, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(le_gap)

## WHO

### Smoking

**Age-standardized current tobacco smoking prevalence (%)** - Percentage of population aged 15+ who currently smoke any tobacco product, age-standardized for cross-country comparison.

**Indicator Code**: M_Est_smk_curr_std  
**Relevance**: Historically, men have had significantly higher smoking rates than women. Smoking is a major contributor to cardiovascular disease, lung cancer, and respiratory diseases. As smoking rates have converged between genders in some countries, the life expectancy gap has narrowed, suggesting smoking is one of the most important modifiable factors contributing to the HALE gender gap.

In [ ]:
filename = '../data/who_smoking_data.csv'
smoking, years = load_and_inventory(filename)

In [ ]:
smoking.head()

In [ ]:
col = 'SmokingPrevalence'
smoking_gap = summarize_gap(smoking, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(smoking_gap)

### Suicide (WHO - For Reference Only)

**NOTE**: This WHO indicator is kept for reference and comparison, but **the IHME Self-Harm indicator (see below) is used in the model** for better temporal coverage (1990-2023 vs 2000-2021 for WHO).

**Age-standardized suicide rates (per 100,000 population)** - Deaths from intentional self-harm, age-standardized for cross-country comparison.

**Indicator Code**: MH_12  
**Relevance**: Suicide rates are typically higher in men across most countries, directly contributing to the gender gap in mortality. Suicide reflects mental health and social factors that differentially affect men and women, and is strongly linked to mortality.

In [ ]:
filename = '../data/who_suicide_rates.csv'
suicide, years = load_and_inventory(filename)

In [ ]:
suicide.head()

In [ ]:
col = 'SuicideRate'
suicide = suicide.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
suicide_gap = summarize_gap(suicide, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(suicide_gap)

### Alcohol (WHO - For Reference Only)

**NOTE**: This WHO indicator is kept for reference and comparison, but **the IHME Alcohol Use Disorders indicator (see below) is used in the model** for better temporal coverage (1990-2023 vs 2019 only for WHO).

**Alcohol-attributable all-cause deaths per 100,000 (age-standardized)** - Deaths from all causes that are attributable to alcohol consumption, including direct alcohol-related deaths and alcohol-attributable deaths from other causes (e.g., accidents, liver disease).

**Indicator Code**: SA_0000001832  
**Relevance**: Men typically have higher rates of alcohol consumption and alcohol-related diseases. Alcohol contributes to liver disease, accidents, and various health conditions, directly impacting mortality. Age-standardized rates match HALE methodology for cross-country comparison.

In [ ]:
filename = '../data/who_alcohol_death_rates.csv'
alcohol, years = load_and_inventory(filename)

In [ ]:
alcohol.head()

In [ ]:
col = 'AlcoholDeathRate'
alcohol = alcohol.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
alcohol_gap = summarize_gap(alcohol, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(alcohol_gap)

### Poison (WHO - For Reference Only)

**NOTE**: This WHO indicator is kept for reference and comparison, but **not used in the model**. The IHME Drug Use Disorders indicator (see below) is used instead, as it provides better temporal coverage (1990-2023 vs 2000-2021 for WHO) and captures drug overdose deaths more comprehensively.

**Mortality rate attributed to unintentional poisoning (per 100,000 population)** - Deaths from accidental poisonings from chemicals, drugs, and other substances.

**Indicator Code**: SDGPOISON  
**Relevance**: Men often have higher rates of accidental deaths, including poisonings. This reflects occupational hazards and risk-taking behaviors that contribute to the gender gap in mortality. Has excellent temporal coverage (2000-2021) and country coverage (196 countries).

In [ ]:
filename = '../data/who_poisoning_rates.csv'
poison, years = load_and_inventory(filename)

In [ ]:
poison.head()

In [ ]:
col = 'PoisoningRate'
poison = poison.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
poison_gap = summarize_gap(poison, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(poison_gap)

### Traffic (WHO - For Reference Only)

**NOTE**: This WHO indicator is kept for reference and comparison, but **the IHME Road Injuries indicator (see below) is used in the model** for much better temporal coverage (1990-2023 vs 2019 only for WHO).

**Road traffic crash deaths, age-standardized death rates (15+), per 100,000 population** - Deaths from road traffic accidents, age-standardized for ages 15+.

**Indicator Code**: SA_0000001459  
**Relevance**: Road traffic deaths are typically 2-4 times higher in men across most countries, making it a major contributor to the gender gap in mortality. Reflects higher exposure to driving (including occupational exposure), occupational hazards, and potentially risk-taking behaviors. Age-standardized rates for ages 15+ match HALE methodology.

In [ ]:
filename = '../data/who_road_traffic_death_rates.csv'
traffic, years = load_and_inventory(filename)

In [ ]:
traffic.head()

In [ ]:
col = 'RoadTrafficDeathRate'
traffic = traffic.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
traffic_gap = summarize_gap(traffic, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(traffic_gap)

### Maternal mortality

**Maternal mortality ratio (per 100,000 live births)** - Deaths of women during pregnancy, childbirth, or within 42 days of termination of pregnancy, per 100,000 live births.

**Indicator Code**: MDG_0000000026  
**Relevance**: Critical for understanding cases where the HALE gender gap is small due to high female mortality, especially in lower-income countries. High maternal mortality can significantly reduce the HALE gender gap by lowering female life expectancy. Inherently female-specific, so only female values are used in analysis.

In [ ]:
filename = '../data/who_maternal_mortality_ratio.csv'
maternal, years = load_and_inventory(filename)

In [ ]:
maternal.head()

In [ ]:
col = 'MaternalMortalityRatio'
maternal = maternal.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
maternal_gap = summarize_gap(maternal, col, sexes=['Female'], cutoff_year=cutoff_year)

In [ ]:
plot_distributions(maternal_gap)

### Homicide (WHO - For Reference Only)

**NOTE**: This WHO indicator is kept for reference and comparison, but **the IHME Interpersonal Violence indicator (see below) is used in the model** for better temporal coverage (1990-2023 vs 2000-2021 for WHO).

**Estimates of rates of homicides per 100,000 population** - Deaths from intentional homicide, including estimates with confidence intervals.

**Indicator Code**: VIOLENCE_HOMICIDERATE  
**Relevance**: Homicide rates are typically much higher in men across most countries, making it a major contributor to the gender gap in mortality. Homicide reflects violence, conflict, and social factors that differentially affect men and women. Has excellent temporal coverage (2000-2021) and country coverage (196 countries).

In [ ]:
filename = '../data/who_homicide_rates.csv'
homicide, years = load_and_inventory(filename)

In [ ]:
homicide.head()

In [ ]:
col = 'HomicideRate'
homicide = homicide.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
homicide_gap = summarize_gap(homicide, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(homicide_gap)

### Intimate Partner Violence

**Proportion of ever-partnered women and girls aged 15-49 years subjected to physical and/or sexual violence by a current or former intimate partner in the previous 12 months (%)** - Prevalence indicator measuring the percentage of women experiencing intimate partner violence.

**Indicator Code**: SDGIPV  
**Relevance**: Note: This is a **prevalence indicator** (percentage), not a direct death rate. IPV affects women's health indirectly through mental health impacts, injuries, and other health consequences. It may contribute to the gender gap in HALE through its effects on women's physical and mental health, though the relationship is complex and indirect. Inherently female-specific, so only female values are used in analysis.

In [ ]:
filename = '../data/who_ipv_prevalence.csv'
ipv, years = load_and_inventory(filename)

In [ ]:
ipv.head()

In [ ]:
col = 'IPVPrevalence'
ipv_gap = summarize_gap(ipv, col, sexes=['Female'], cutoff_year=cutoff_year)

In [ ]:
plot_distributions(ipv_gap)

### Under five mortality rate (WHO - For Reference Only)

**NOTE**: This WHO indicator is kept for reference and comparison, but **the IHME All-Cause Deaths Under 5 Years indicator (see below) is used in the model** for better temporal coverage (1990-2023 vs 1932-2023 for WHO, but IHME provides consistent methodology with other IHME indicators).

**Under-five mortality rate (probability of dying by age 5 per 1000 live births)** - Deaths of children under age 5 per 1,000 live births, with gender breakdowns.

**Indicator Code**: MDG_0000000007  
**Relevance**: HALE is calculated from birth, so under-five mortality directly affects HALE calculations. If child mortality differs by gender, it directly contributes to the HALE gender gap. Infant mortality is typically higher in males (biological vulnerability + some behavioral factors). More important in lower-income countries with high child mortality. Note: MDG_0000000007 chosen over u5mr for better data quality when filtered for sex dimension.

In [ ]:
filename = '../data/who_u5mr.csv'
u5mr, years = load_and_inventory(filename)

In [ ]:
u5mr.head()

In [ ]:
col = 'U5MR'
u5mr = u5mr.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
u5mr_gap = summarize_gap(u5mr, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(u5mr_gap)

### Cardiovascular Disease

**Age-standardized cardiovascular disease death rates (per 100,000)** - Deaths from cardiovascular diseases (heart disease, stroke, etc.), age-standardized for cross-country comparison.

**Indicator Code**: Multiple codes tried (WHS2_161, etc.) - see `who_data.py` for implementation details  
**Relevance**: Men typically have higher rates of cardiovascular disease and heart attacks, contributing significantly to the gender gap in mortality. Risk factors include smoking, diet, and potentially biological differences. May capture effects of smoking and other risk factors. Age-standardized rates match HALE methodology.

In [ ]:
filename = '../data/who_cardiovascular_death_rates.csv'
cardio, years = load_and_inventory(filename)

In [ ]:
# Rename DeathRate to CardioDeathRate to avoid ambiguity
cardio = cardio.rename(columns={'DeathRate': 'CardioDeathRate'})
cardio.head()

In [ ]:
col = 'CardioDeathRate'
cardio_gap = summarize_gap(cardio, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(cardio_gap)

### Diabetes

**Age-standardized death rates, diabetes mellitus (per 100,000)** - Deaths from diabetes, age-standardized for cross-country comparison.

**Indicator Code**: SA_0000001440  
**Relevance**: Diabetes is a chronic condition that can contribute to the gender gap in mortality, though the relationship may vary by country and healthcare access. Age-standardized rates match HALE methodology. **Limitation**: Only has data for 2004 (similar to cardiovascular disease indicators), which limits temporal analysis but provides a good cross-sectional snapshot.

In [ ]:
filename = '../data/who_diabetes_death_rates.csv'
diabetes, years = load_and_inventory(filename)

In [ ]:
diabetes.head()

In [ ]:
col = 'DiabetesDeathRate'
diabetes_gap = summarize_gap(diabetes, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(diabetes_gap)

### NCD Mortality (30-70 years)

**Probability (%) of dying between age 30 and exact age 70 from any of cardiovascular disease, cancer, diabetes, or chronic respiratory disease** - Combined non-communicable disease mortality indicator.

**Indicator Code**: NCDMORT3070  
**Relevance**: Combines multiple causes of death (cardiovascular disease, cancer, diabetes, chronic respiratory disease), so it's less specific than individual cause indicators. However, it has much better temporal coverage (2000-2021) than diabetes-specific indicators (which only have 2004 data). This makes it useful for model comparison - trading off specificity for temporal coverage. The combined indicator may capture overall NCD mortality patterns that contribute to the HALE gender gap.

In [ ]:
filename = '../data/who_ncd_mortality_30_70.csv'
ncdmort, years = load_and_inventory(filename)

In [ ]:
ncdmort.head()

In [ ]:
col = 'NCDMortality30_70'
ncdmort_gap = summarize_gap(ncdmort, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(ncdmort_gap)

## IHME

### Drug Use Disorders (IHME) - USED IN MODEL

**Drug use disorder death rates (per 100,000 population)** - Deaths from drug use disorders, including overdoses, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Drug overdoses, particularly opioid overdoses, are a major cause of death in some OECD countries (especially the US) and may contribute significantly to the HALE gender gap. **This IHME indicator is used in the model** instead of WHO Poisoning because it provides better temporal coverage (1990-2023 vs 2000-2021 for WHO) and captures drug overdose deaths more comprehensively. This indicator captures overdose deaths that may not be fully captured in the WHO poisoning indicator. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
def load_ihme_indicator(filename_male, filename_female, value_col_name, indicator_code, indicator_name):
    """
    Load IHME indicator data from separate male and female files and convert to WHO-compatible format.
    
    Converts IHME CSV format (Location=country name, Sex="Male"/"Female") to WHO format
    (Country=country code, Sex="Male"/"Female", etc.). Filters to 2000-2019.
    
    Parameters
    ----------
    filename_male : str
        Path to the IHME CSV file with male data.
    filename_female : str
        Path to the IHME CSV file with female data.
    value_col_name : str
        Name for the value column (e.g., 'DrugDisorderDeathRate', 'DiabetesDeathRate').
    indicator_code : str
        Indicator code for the IHME indicator (e.g., 'IHME_DRUG_DISORDERS', 'IHME_DIABETES_TYPE2').
    indicator_name : str
        Human-readable indicator name (e.g., 'Drug use disorders, death rate per 100,000').
        
    Returns
    -------
    df : pandas.DataFrame
        DataFrame in WHO-compatible format with columns: IndicatorCode, IndicatorName,
        Code, CountryCode, Year, Sex, value_col_name, value_col_name_Low, 
        value_col_name_High, Country.
    years : numpy.ndarray
        Array of unique years present in the filtered dataset.
    """
    # Create reverse mapping from country name to code
    who_country_to_code = {country: code for code, country in code_to_who_country.items()}
    
    # Map IHME country names that differ from WHO names
    ihme_country_name_mapping = {
        'Republic of Korea': 'South Korea',
        'United States of America': 'United States'
    }
    
    def process_ihme_file(filename, sex_value):
        """Helper function to process a single IHME file."""
        df = pd.read_csv(filename)
        
        # Filter to 2000-2019 (exclude 2020+ for COVID-19 reasons)
        df = df.query('Year >= 2000 and Year <= 2019')
        
        # Filter to "All ages" (if Age column exists)
        if 'Age' in df.columns:
            df = df.query('Age == "All ages"')
        
        # Map IHME country names to WHO country names
        df['Location'] = df['Location'].replace(ihme_country_name_mapping)
        
        # Convert country names to codes
        df['Code'] = df['Location'].map(who_country_to_code)
        
        # Filter out rows where country mapping failed (not in our country list)
        df = df[df['Code'].notna()].copy()
        
        # Set Sex column to the specified value (Male or Female)
        df['Sex'] = sex_value
        
        # Rename and create columns to match WHO format
        df['IndicatorCode'] = indicator_code
        df['IndicatorName'] = indicator_name
        df['CountryCode'] = 'COUNTRY'
        df[value_col_name] = df['Value']
        df[f'{value_col_name}_Low'] = df['Lower bound']
        df[f'{value_col_name}_High'] = df['Upper bound']
        df['Country'] = df['Location']
        
        # Select and reorder columns to match WHO format
        columns_to_keep = [
            'IndicatorCode', 'IndicatorName', 'Code', 'CountryCode', 'Year', 'Sex',
            value_col_name, f'{value_col_name}_Low', f'{value_col_name}_High',
            'Country'
        ]
        df = df[columns_to_keep].copy()
        
        return df
    
    # Load and process both files
    df_male = process_ihme_file(filename_male, 'Male')
    df_female = process_ihme_file(filename_female, 'Female')
    
    # Concatenate male and female data
    df = pd.concat([df_male, df_female], ignore_index=True)
    
    # Sort by country, sex, and year
    df = df.sort_values(['Country', 'Sex', 'Year']).reset_index(drop=True)
    
    years = df['Year'].unique()
    
    print(df.shape)
    print(f"Years: {years.min():.0f} - {years.max():.0f}")
    print(f"Countries: {df['Country'].nunique()}")
    print(f"Sex categories: {df['Sex'].unique()}")
    
    return df, years

In [ ]:
filename_male = '../data/ihme_drug_disorder_deaths_male.csv'
filename_female = '../data/ihme_drug_disorder_deaths_female.csv'
drug_disorders, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='DrugDisorderDeathRate',
    indicator_code='IHME_DRUG_DISORDERS',
    indicator_name='Drug use disorders, death rate per 100,000'
)

In [ ]:
drug_disorders.head()

In [ ]:
col = 'DrugDisorderDeathRate'
drug_disorders = drug_disorders.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
drug_disorders_gap = summarize_gap(drug_disorders, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(drug_disorders_gap)

### Diabetes Type 2 (IHME)

**Diabetes type 2 death rates (per 100,000 population)** - Deaths from diabetes mellitus type 2, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: This is an alternative to the WHO diabetes death rate indicator (SA_0000001440) which only has data for 2004. IHME data may have better temporal coverage, allowing for more recent data to be used in the analysis. Diabetes is a chronic condition that can contribute to the gender gap in mortality, though the relationship may vary by country and healthcare access. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_diabetes_deaths_male.csv'
filename_female = '../data/ihme_diabetes_deaths_female.csv'
diabetes_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='DiabetesDeathRate',
    indicator_code='IHME_DIABETES_TYPE2',
    indicator_name='Diabetes mellitus type 2, death rate per 100,000'
)

In [ ]:
diabetes_ihme.head()

In [ ]:
col = 'DiabetesDeathRate'
diabetes_ihme = diabetes_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
diabetes_ihme_gap = summarize_gap(diabetes_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(diabetes_ihme_gap)

### Cardiovascular Diseases (IHME)

**Cardiovascular diseases death rates (per 100,000 population)** - Deaths from cardiovascular diseases, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Cardiovascular diseases are a major cause of death and may contribute significantly to the HALE gender gap. This is an alternative to the WHO cardiovascular disease death rate indicators which only have data for 2004. IHME data may have better temporal coverage, allowing for more recent data to be used in the analysis. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_cardiovascular_deaths_male.csv'
filename_female = '../data/ihme_cardiovascular_deaths_female.csv'
cardiovascular_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='CardioDeathRate',
    indicator_code='IHME_CARDIOVASCULAR',
    indicator_name='Cardiovascular diseases, death rate per 100,000'
)

In [ ]:
cardiovascular_ihme.head()

In [ ]:
col = 'CardioDeathRate'
cardiovascular_ihme = cardiovascular_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
cardiovascular_ihme_gap = summarize_gap(cardiovascular_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(cardiovascular_ihme_gap)

### Neoplasms (Cancer) (IHME)

**Neoplasms (cancer) death rates (per 100,000 population)** - Deaths from neoplasms (cancer), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Neoplasms (cancer) are a major cause of death and may contribute significantly to the HALE gender gap. Different types of cancer have different gender patterns (e.g., lung cancer is often higher in men, breast cancer is female-specific). This indicator provides comprehensive cancer death rates with better temporal coverage than WHO indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_neoplasms_deaths_male.csv'
filename_female = '../data/ihme_neoplasms_deaths_female.csv'
neoplasms_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='NeoplasmsDeathRate',
    indicator_code='IHME_NEOPLASMS',
    indicator_name='Neoplasms (cancer), death rate per 100,000'
)

In [ ]:
neoplasms_ihme.head()

In [ ]:
col = 'NeoplasmsDeathRate'
neoplasms_ihme = neoplasms_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
neoplasms_ihme_gap = summarize_gap(neoplasms_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(neoplasms_ihme_gap)

### Chronic Respiratory Diseases (IHME)

**Chronic respiratory diseases death rates (per 100,000 population)** - Deaths from chronic respiratory diseases (including COPD, asthma, and other chronic lung conditions), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Chronic respiratory diseases are a major cause of death and may contribute significantly to the HALE gender gap. These diseases often have gender differences due to factors such as smoking patterns, occupational exposures, and environmental factors. This indicator provides comprehensive chronic respiratory disease death rates with better temporal coverage than WHO indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_chronic_respiratory_deaths_male.csv'
filename_female = '../data/ihme_chronic_respiratory_deaths_female.csv'
chronic_respiratory_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='ChronicRespiratoryDeathRate',
    indicator_code='IHME_CHRONIC_RESPIRATORY',
    indicator_name='Chronic respiratory diseases, death rate per 100,000'
)

In [ ]:
chronic_respiratory_ihme.head()

In [ ]:
col = 'ChronicRespiratoryDeathRate'
chronic_respiratory_ihme = chronic_respiratory_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
chronic_respiratory_ihme_gap = summarize_gap(chronic_respiratory_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(chronic_respiratory_ihme_gap)

### Liver Disease (Cirrhosis and Other Chronic Liver Diseases) (IHME)

**Liver disease death rates (per 100,000 population)** - Deaths from cirrhosis and other chronic liver diseases, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Liver disease (cirrhosis and other chronic liver diseases) is a significant cause of death and may contribute to the HALE gender gap. Men typically have higher rates of liver disease mortality than women, often due to higher alcohol consumption, hepatitis infections, and other risk factors. This indicator provides comprehensive liver disease death rates with excellent temporal coverage (1990-2023, 34 years) and good country coverage. Liver disease is often related to alcohol consumption, but also includes non-alcoholic causes such as viral hepatitis, non-alcoholic fatty liver disease, and other chronic liver conditions. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_liver_disease_deaths_male.csv'
filename_female = '../data/ihme_liver_disease_deaths_female.csv'
liver_disease_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='LiverDiseaseDeathRate',
    indicator_code='IHME_LIVER_DISEASE',
    indicator_name='Cirrhosis and other chronic liver diseases, death rate per 100,000'
)

In [ ]:
liver_disease_ihme.head()

In [ ]:
col = 'LiverDiseaseDeathRate'
liver_disease_ihme = liver_disease_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
liver_disease_ihme_gap = summarize_gap(liver_disease_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(liver_disease_ihme_gap)

### Unintentional Injuries (IHME)

**Unintentional injuries death rates (per 100,000 population)** - Deaths from unintentional injuries (including falls, drowning, fires, and other accidents), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Unintentional injuries are a significant cause of death and may contribute to the HALE gender gap. These injuries often show gender differences due to occupational exposures, risk-taking behaviors, and activity patterns. This indicator provides comprehensive unintentional injury death rates with better temporal coverage (1990-2023) than many WHO indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_unintentional_injuries_deaths_male.csv'
filename_female = '../data/ihme_unintentional_injuries_deaths_female.csv'
unintentional_injuries_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='UnintentionalInjuriesDeathRate',
    indicator_code='IHME_UNINTENTIONAL_INJURIES',
    indicator_name='Unintentional injuries, death rate per 100,000'
)

In [ ]:
unintentional_injuries_ihme.head()

In [ ]:
col = 'UnintentionalInjuriesDeathRate'
unintentional_injuries_ihme = unintentional_injuries_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
unintentional_injuries_ihme_gap = summarize_gap(unintentional_injuries_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(unintentional_injuries_ihme_gap)

### Alcohol Use Disorders (IHME) - USED IN MODEL

**Alcohol use disorders death rates (per 100,000 population)** - Deaths from alcohol use disorders, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Alcohol use disorders are a significant cause of death and may contribute to the HALE gender gap. Men typically have higher rates of alcohol-related mortality than women. This indicator provides comprehensive alcohol use disorder death rates with excellent temporal coverage (1990-2023, 34 years) and good country coverage (40 countries). This is an alternative to the WHO alcohol-attributable death rate indicator (SA_0000001832) which only has data for 2019. **This IHME version is used in the model** because it provides much better temporal coverage, allowing for temporal analysis and more recent data. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_alcohol_use_disorders_deaths_male.csv'
filename_female = '../data/ihme_alcohol_use_disorders_deaths_female.csv'
alcohol_use_disorders_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='AlcoholUseDisordersDeathRate',
    indicator_code='IHME_ALCOHOL_USE_DISORDERS',
    indicator_name='Alcohol use disorders, death rate per 100,000'
)

In [ ]:
alcohol_use_disorders_ihme.head()

In [ ]:
col = 'AlcoholUseDisordersDeathRate'
alcohol_use_disorders_ihme = alcohol_use_disorders_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
alcohol_use_disorders_ihme_gap = summarize_gap(alcohol_use_disorders_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(alcohol_use_disorders_ihme_gap)

### Self-Harm (IHME) - USED IN MODEL

**Self-harm (suicide) death rates (per 100,000 population)** - Deaths from self-harm (suicide), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Self-harm (suicide) is a significant cause of death and contributes to the HALE gender gap. Men typically have much higher suicide rates than women in most countries. This indicator provides comprehensive self-harm death rates with excellent temporal coverage (1990-2023, 34 years) and good country coverage (40 countries). This is an alternative to the WHO suicide rate indicator (MH_12) which has data for 2000-2021. **This IHME version is used in the model** because it provides better temporal coverage (starting from 1990) and consistent methodology with other IHME indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_self_harm_deaths_male.csv'
filename_female = '../data/ihme_self_harm_deaths_female.csv'
self_harm_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='SelfHarmDeathRate',
    indicator_code='IHME_SELF_HARM',
    indicator_name='Self-harm (suicide), death rate per 100,000'
)

In [ ]:
self_harm_ihme.head()

In [ ]:
col = 'SelfHarmDeathRate'
self_harm_ihme = self_harm_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
self_harm_ihme_gap = summarize_gap(self_harm_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(self_harm_ihme_gap)

### Interpersonal Violence (IHME) - USED IN MODEL

**Interpersonal violence (homicide) death rates (per 100,000 population)** - Deaths from interpersonal violence (homicide), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Interpersonal violence (homicide) is a significant cause of death and contributes to the HALE gender gap. Men typically have much higher homicide rates than women in most countries. This indicator provides comprehensive interpersonal violence death rates with excellent temporal coverage (1990-2023, 34 years) and good country coverage (40 countries). This is an alternative to the WHO homicide rate indicator (VIOLENCE_HOMICIDERATE) which has data for 2000-2021. **This IHME version is used in the model** because it provides better temporal coverage (starting from 1990) and consistent methodology with other IHME indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_interpersonal_violence_deaths_male.csv'
filename_female = '../data/ihme_interpersonal_violence_deaths_female.csv'
interpersonal_violence_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='InterpersonalViolenceDeathRate',
    indicator_code='IHME_INTERPERSONAL_VIOLENCE',
    indicator_name='Interpersonal violence (homicide), death rate per 100,000'
)

In [ ]:
interpersonal_violence_ihme.head()

In [ ]:
col = 'InterpersonalViolenceDeathRate'
interpersonal_violence_ihme = interpersonal_violence_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
interpersonal_violence_ihme_gap = summarize_gap(interpersonal_violence_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(interpersonal_violence_ihme_gap)

### Road Injuries (IHME) - USED IN MODEL

**Road injuries (road traffic crash) death rates (per 100,000 population)** - Deaths from road injuries (road traffic crashes), from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Road injuries (road traffic crashes) are a significant cause of death and contribute to the HALE gender gap. Men typically have 2-4 times higher road traffic death rates than women in most countries due to higher exposure to driving (including occupational exposure), occupational hazards, and potentially risk-taking behaviors. This indicator provides comprehensive road injury death rates with excellent temporal coverage (1990-2023, 34 years) and good country coverage (40 countries). This is an alternative to the WHO road traffic crash death rate indicator (SA_0000001459) which only has data for 2019. **This IHME version is used in the model** because it provides much better temporal coverage (1990-2023 vs 2019 only) and consistent methodology with other IHME indicators. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_road_injuries_deaths_male.csv'
filename_female = '../data/ihme_road_injuries_deaths_female.csv'
road_injuries_ihme, years = load_ihme_indicator(
    filename_male, filename_female,
    value_col_name='RoadInjuriesDeathRate',
    indicator_code='IHME_ROAD_INJURIES',
    indicator_name='Road injuries (road traffic crashes), death rate per 100,000'
)

In [ ]:
road_injuries_ihme.head()

In [ ]:
col = 'RoadInjuriesDeathRate'
road_injuries_ihme = road_injuries_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
road_injuries_ihme_gap = summarize_gap(road_injuries_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(road_injuries_ihme_gap)

### Maternal Disorders (IHME)

**Maternal disorders death rates (per 100,000 population)** - Deaths from maternal disorders, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: Maternal disorders (maternal mortality) are a significant cause of death for women and can contribute to the HALE gender gap, especially in lower-income countries. High maternal mortality can significantly reduce female life expectancy, explaining why some countries have smaller gender gaps. This indicator provides comprehensive maternal disorder death rates with excellent temporal coverage (1990-2023, 34 years) and good country coverage (40 countries). This is an alternative to the WHO maternal mortality ratio indicator (MDG_0000000026) which has data for 1985-2023. Note: WHO indicator uses ratio per 100,000 live births, while IHME uses rate per 100,000 population, so they measure slightly different things. IHME data may be useful for temporal analysis and provides consistent methodology with other IHME indicators. Inherently female-specific, so only female values are used in analysis.

In [ ]:
filename_female = '../data/ihme_maternal_disorders_deaths_female.csv'
# Maternal disorders is female-only, so we need to handle it differently
# Load the female file and create a compatible format
maternal_disorders_ihme_female = pd.read_csv(filename_female)

# Filter to 2000-2019 (exclude 2020+ for COVID-19 reasons)
maternal_disorders_ihme_female = maternal_disorders_ihme_female.query('Year >= 2000 and Year <= 2019')

# Filter to "All ages" (if Age column exists)
if 'Age' in maternal_disorders_ihme_female.columns:
    maternal_disorders_ihme_female = maternal_disorders_ihme_female.query('Age == "All ages"')

# Map IHME country names to WHO country names
who_country_to_code = {country: code for code, country in code_to_who_country.items()}
ihme_country_name_mapping = {
    'Republic of Korea': 'South Korea',
    'United States of America': 'United States'
}
maternal_disorders_ihme_female['Location'] = maternal_disorders_ihme_female['Location'].replace(ihme_country_name_mapping)

# Convert country names to codes
maternal_disorders_ihme_female['Code'] = maternal_disorders_ihme_female['Location'].map(who_country_to_code)

# Filter out rows where country mapping failed
maternal_disorders_ihme_female = maternal_disorders_ihme_female[maternal_disorders_ihme_female['Code'].notna()].copy()

# Set Sex column to Female
maternal_disorders_ihme_female['Sex'] = 'Female'

# Rename and create columns to match WHO format
maternal_disorders_ihme_female['IndicatorCode'] = 'IHME_MATERNAL_DISORDERS'
maternal_disorders_ihme_female['IndicatorName'] = 'Maternal disorders, death rate per 100,000'
maternal_disorders_ihme_female['CountryCode'] = 'COUNTRY'
maternal_disorders_ihme_female['MaternalDisordersDeathRate'] = maternal_disorders_ihme_female['Value']
maternal_disorders_ihme_female['MaternalDisordersDeathRate_Low'] = maternal_disorders_ihme_female['Lower bound']
maternal_disorders_ihme_female['MaternalDisordersDeathRate_High'] = maternal_disorders_ihme_female['Upper bound']
maternal_disorders_ihme_female['Country'] = maternal_disorders_ihme_female['Location']

# Select and reorder columns to match WHO format
columns_to_keep = [
    'IndicatorCode', 'IndicatorName', 'Code', 'CountryCode', 'Year', 'Sex',
    'MaternalDisordersDeathRate', 'MaternalDisordersDeathRate_Low', 'MaternalDisordersDeathRate_High',
    'Country'
]
maternal_disorders_ihme = maternal_disorders_ihme_female[columns_to_keep].copy()

# Sort by country and year
maternal_disorders_ihme = maternal_disorders_ihme.sort_values(['Country', 'Year']).reset_index(drop=True)

years = maternal_disorders_ihme['Year'].unique()

print(maternal_disorders_ihme.shape)
print(f"Years: {years.min():.0f} - {years.max():.0f}")
print(f"Countries: {maternal_disorders_ihme['Country'].nunique()}")
print(f"Sex categories: {maternal_disorders_ihme['Sex'].unique()}")

In [ ]:
maternal_disorders_ihme.head()

In [ ]:
col = 'MaternalDisordersDeathRate'
maternal_disorders_ihme = maternal_disorders_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
maternal_disorders_ihme_gap = summarize_gap(maternal_disorders_ihme, col, sexes=['Female'], cutoff_year=cutoff_year)

In [ ]:
plot_distributions(maternal_disorders_ihme_gap)

### All-Cause Deaths Under 5 Years of Age (IHME) - USED IN MODEL

**All-cause deaths under 5 years of age (per 100,000 population)** - Deaths from all causes for children under 5 years of age, from IHME Global Burden of Disease data.

**Data Source**: IHME Global Burden of Disease (https://vizhub.healthdata.org/gbd-compare/)  
**Relevance**: All-cause mortality for children under 5 years of age is relevant to the HALE gender gap because HALE is calculated from birth, so early-life mortality directly affects HALE calculations. If child mortality differs by gender, it directly contributes to the HALE gender gap. Infant and child mortality is typically higher in males (biological vulnerability + some behavioral factors). **This IHME version is used in the model** because it provides better temporal coverage (1990-2023) and consistent methodology with other IHME indicators. This is different from the WHO under-five mortality rate (U5MR, MDG_0000000007) which measures deaths per 1,000 live births. The IHME indicator measures deaths per 100,000 population, providing a complementary perspective on early-life mortality. Data includes separate male and female values, allowing for gender gap analysis.

In [ ]:
filename_male = '../data/ihme_all_causes_under5_deaths_male.csv'
filename_female = '../data/ihme_all_causes_under5_deaths_female.csv'
# All-cause under 5 needs special handling because it's for "<5 years" age group, not "All ages"
# We'll create a modified version of load_ihme_indicator that filters for "<5 years" instead

def load_ihme_indicator_under5(filename_male, filename_female, value_col_name, indicator_code, indicator_name):
    """
    Load IHME indicator data for under-5 age group from separate male and female files.
    
    Similar to load_ihme_indicator but filters for "<5 years" age group instead of "All ages".
    """
    # Create reverse mapping from country name to code
    who_country_to_code = {country: code for code, country in code_to_who_country.items()}
    
    # Map IHME country names that differ from WHO names
    ihme_country_name_mapping = {
        'Republic of Korea': 'South Korea',
        'United States of America': 'United States'
    }
    
    def process_ihme_file(filename, sex_value):
        """Helper function to process a single IHME file."""
        df = pd.read_csv(filename)
        
        # Filter to 2000-2019 (exclude 2020+ for COVID-19 reasons)
        df = df.query('Year >= 2000 and Year <= 2019')
        
        # Filter to "<5 years" age group
        if 'Age' in df.columns:
            df = df.query('Age == "<5 years"')
        
        # Map IHME country names to WHO country names
        df['Location'] = df['Location'].replace(ihme_country_name_mapping)
        
        # Convert country names to codes
        df['Code'] = df['Location'].map(who_country_to_code)
        
        # Filter out rows where country mapping failed (not in our country list)
        df = df[df['Code'].notna()].copy()
        
        # Set Sex column to the specified value (Male or Female)
        df['Sex'] = sex_value
        
        # Rename and create columns to match WHO format
        df['IndicatorCode'] = indicator_code
        df['IndicatorName'] = indicator_name
        df['CountryCode'] = 'COUNTRY'
        df[value_col_name] = df['Value']
        df[f'{value_col_name}_Low'] = df['Lower bound']
        df[f'{value_col_name}_High'] = df['Upper bound']
        df['Country'] = df['Location']
        
        # Select and reorder columns to match WHO format
        columns_to_keep = [
            'IndicatorCode', 'IndicatorName', 'Code', 'CountryCode', 'Year', 'Sex',
            value_col_name, f'{value_col_name}_Low', f'{value_col_name}_High',
            'Country'
        ]
        df = df[columns_to_keep].copy()
        
        return df
    
    # Load and process both files
    df_male = process_ihme_file(filename_male, 'Male')
    df_female = process_ihme_file(filename_female, 'Female')
    
    # Concatenate male and female data
    df = pd.concat([df_male, df_female], ignore_index=True)
    
    # Sort by country, sex, and year
    df = df.sort_values(['Country', 'Sex', 'Year']).reset_index(drop=True)
    
    years = df['Year'].unique()
    
    print(df.shape)
    print(f"Years: {years.min():.0f} - {years.max():.0f}")
    print(f"Countries: {df['Country'].nunique()}")
    print(f"Sex categories: {df['Sex'].unique()}")
    
    return df, years

all_causes_under5_ihme, years = load_ihme_indicator_under5(
    filename_male, filename_female,
    value_col_name='AllCausesUnder5DeathRate',
    indicator_code='IHME_ALL_CAUSES_UNDER5',
    indicator_name='All-cause deaths under 5 years, death rate per 100,000'
)

In [ ]:
all_causes_under5_ihme.head()

In [ ]:
col = 'AllCausesUnder5DeathRate'
all_causes_under5_ihme = all_causes_under5_ihme.rename(columns=column_name_mapping)
col = column_name_mapping.get(col, col)
all_causes_under5_ihme_gap = summarize_gap(all_causes_under5_ihme, col, cutoff_year=cutoff_year)

In [ ]:
plot_distributions(all_causes_under5_ihme_gap)

## Comparing WHO and IHME Indicators

This section compares indicators where we have data from both WHO and IHME sources. For each indicator, we show side-by-side scatter plots comparing the gender gap values, using the same axis scaling to facilitate comparison.

In [ ]:
# Function to create side-by-side comparison plots
def compare_who_ihme(who_df, ihme_df, who_col, ihme_col, who_label, ihme_label, title):
    """
    Create side-by-side scatter plots comparing WHO and IHME gap values.
    
    Parameters
    ----------
    who_df : pandas.DataFrame
        DataFrame with WHO data, indexed by country code
    ihme_df : pandas.DataFrame
        DataFrame with IHME data, indexed by country code
    who_col : str
        Column name for WHO gap values (e.g., 'Gap_Alcohol')
    ihme_col : str
        Column name for IHME gap values (e.g., 'Gap_Alcohol')
    who_label : str
        Label for WHO indicator
    ihme_label : str
        Label for IHME indicator
    title : str
        Overall title for the comparison
    """
    # Get OECD countries for both datasets
    who_oecd = get_oecd(who_df)
    ihme_oecd = get_oecd(ihme_df)
    
    # Find common countries
    common_countries = who_oecd.index.intersection(ihme_oecd.index)
    who_common = who_oecd.loc[common_countries]
    ihme_common = ihme_oecd.loc[common_countries]
    
    # Get all values to determine domain
    all_values = []
    if who_col in who_common.columns:
        all_values.extend(who_common[who_col].dropna().tolist())
    if ihme_col in ihme_common.columns:
        all_values.extend(ihme_common[ihme_col].dropna().tolist())
    
    if not all_values:
        print(f"No data available for {title}")
        return
    
    domain = [0, max(all_values) * 1.05]  # Add 5% padding
    
    # Create side-by-side distribution plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot WHO distribution
    plt.sca(ax1)
    if who_col in who_common.columns:
        who_values = who_common[who_col].dropna()
        # Get Male and Female columns for scatter plot
        who_male_col = who_col.replace('Gap_', '') + '_Male'
        who_female_col = who_col.replace('Gap_', '') + '_Female'
        
        if who_male_col in who_common.columns and who_female_col in who_common.columns:
            scatter_plot(who_common, [who_male_col, who_female_col], domain,
                        xlabel=f'{who_label} Male',
                        ylabel=f'{who_label} Female',
                        title=f'WHO: {who_label}')
        else:
            ax1.text(0.5, 0.5, f'Missing Male/Female columns', 
                    ha='center', va='center', transform=ax1.transAxes)
            ax1.set_title(f'WHO: {who_label}')
    else:
        ax1.text(0.5, 0.5, f'No {who_col} column', 
                ha='center', va='center', transform=ax1.transAxes)
        ax1.set_title(f'WHO: {who_label}')
    
    # Plot IHME distribution
    plt.sca(ax2)
    if ihme_col in ihme_common.columns:
        ihme_values = ihme_common[ihme_col].dropna()
        # Get Male and Female columns for scatter plot
        ihme_male_col = ihme_col.replace('Gap_', '') + '_Male'
        ihme_female_col = ihme_col.replace('Gap_', '') + '_Female'
        
        if ihme_male_col in ihme_common.columns and ihme_female_col in ihme_common.columns:
            scatter_plot(ihme_common, [ihme_male_col, ihme_female_col], domain,
                        xlabel=f'{ihme_label} Male',
                        ylabel=f'{ihme_label} Female',
                        title=f'IHME: {ihme_label}')
        else:
            ax2.text(0.5, 0.5, f'Missing Male/Female columns', 
                    ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title(f'IHME: {ihme_label}')
    else:
        ax2.text(0.5, 0.5, f'No {ihme_col} column', 
                ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title(f'IHME: {ihme_label}')
    
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Create direct comparison scatter plot (WHO vs IHME)
    if who_col in who_common.columns and ihme_col in ihme_common.columns:
        # Merge on index (country codes)
        comparison_df = pd.DataFrame({
            who_label: who_common[who_col],
            ihme_label: ihme_common[ihme_col]
        })
        comparison_df = comparison_df.dropna()
        
        if len(comparison_df) > 0:
            # Determine domain for comparison plot (use same domain for both axes)
            comp_domain = [0, max(comparison_df.max()) * 1.05]
            
            plt.figure(figsize=(8, 8))
            scatter_plot(comparison_df, [who_label, ihme_label], comp_domain,
                        xlabel=f'WHO {who_label} Gap',
                        ylabel=f'IHME {ihme_label} Gap',
                        title=f'{title}: Gap Comparison')
            plt.show()
        else:
            print(f"No overlapping data for {title}")
    else:
        print(f"Missing columns: WHO={who_col in who_common.columns}, IHME={ihme_col in ihme_common.columns}")

### Alcohol: WHO vs IHME

In [ ]:
compare_who_ihme(
    alcohol_gap, alcohol_use_disorders_ihme_gap,
    'Gap_Alcohol', 'Gap_Alcohol',
    'WHO Alcohol', 'IHME Alcohol Use Disorders',
    'Alcohol: WHO vs IHME Comparison'
)

### Suicide: WHO vs IHME

In [ ]:
compare_who_ihme(
    suicide_gap, self_harm_ihme_gap,
    'Gap_Suicide', 'Gap_Suicide',
    'WHO Suicide', 'IHME Self-Harm',
    'Suicide: WHO vs IHME Comparison'
)

### Homicide: WHO vs IHME

In [ ]:
compare_who_ihme(
    homicide_gap, interpersonal_violence_ihme_gap,
    'Gap_Homicide', 'Gap_Homicide',
    'WHO Homicide', 'IHME Interpersonal Violence',
    'Homicide: WHO vs IHME Comparison'
)

### Road Traffic: WHO vs IHME

In [ ]:
compare_who_ihme(
    traffic_gap, road_injuries_ihme_gap,
    'Gap_RoadTraffic', 'Gap_RoadTraffic',
    'WHO Road Traffic', 'IHME Road Injuries',
    'Road Traffic: WHO vs IHME Comparison'
)

### Poisoning (WHO) vs Drug Disorders (IHME)

Note: These are different concepts (unintentional poisoning vs drug use disorders), but both relate to substance-related mortality.

In [ ]:
compare_who_ihme(
    poison_gap, drug_disorders_gap,
    'Gap_Poisoning', 'Gap_DrugDisorder',
    'WHO Poisoning', 'IHME Drug Disorders',
    'Poisoning (WHO) vs Drug Disorders (IHME) Comparison'
)

### Under-Five Mortality: WHO vs IHME

Note: WHO measures deaths per 1,000 live births, while IHME measures deaths per 100,000 population. These are different measurement units, so direct comparison of absolute values is not meaningful, but the gender gaps can still be compared.

In [ ]:
compare_who_ihme(
    u5mr_gap, all_causes_under5_ihme_gap,
    'Gap_Childhood', 'Gap_Childhood',
    'WHO Under-Five Mortality', 'IHME All-Cause Under 5',
    'Under-Five Mortality: WHO vs IHME Comparison'
)

---

## Data Preparation for Regression Analysis

### Prepare Target Variables (HALE and Life Expectancy Gender Gaps)

In [ ]:
# Calculate HALE gender gap from existing hale_gap DataFrame
# Gap = Female - Male (positive means females have higher HALE)
hale_gap['HALE_gap'] = hale_gap['HALE_Years_Female'] - hale_gap['HALE_Years_Male']

# Filter to OECD countries
hale_oecd = get_oecd(hale_gap)

# Display summary
hale_oecd[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']].describe()

In [ ]:
# Calculate Life Expectancy gender gap from existing le_gap DataFrame
# Gap = Female - Male (positive means females have higher Life Expectancy)
le_gap['LifeExpectancy_gap'] = le_gap['LifeExpectancy_Years_Female'] - le_gap['LifeExpectancy_Years_Male']

# Filter to OECD countries
le_oecd = get_oecd(le_gap)

# Display summary
le_oecd[['LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']].describe()

### Merge All Predictors into Single Dataset

In [ ]:
# Start with HALE and LE
analysis_df = hale_oecd[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']].copy()

analysis_df = analysis_df.join(le_oecd[['LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']], how='outer')

In [ ]:
# Unified mapping from indicator names to complete _gap datasets
# This is the single source of truth for all indicators used in the analysis
indicator_datasets = {
    'Alcohol': alcohol_use_disorders_ihme_gap,  # Using IHME version, renamed to 'Alcohol' via column_name_mapping
    'ChronicRespiratory': chronic_respiratory_ihme_gap,
    'UnintentionalInjury': unintentional_injuries_ihme_gap,
    'RoadTraffic': road_injuries_ihme_gap,  # Using IHME version, renamed to 'RoadTraffic' via column_name_mapping
    'Diabetes': diabetes_ihme_gap,
    'Cardiovascular': cardiovascular_ihme_gap,
    # 'Childhood': all_causes_under5_ihme_gap,  # Removed - low importance and limited temporal coverage. WHO U5MR (per 1,000 live births) is methodologically appropriate but has limited temporal coverage. IHME version (per 100,000 population) is confounded with age structure and fertility.
    'DrugDisorder': drug_disorders_gap,  # Using IHME version, replacing WHO Poisoning
    'Homicide': interpersonal_violence_ihme_gap,  # Using IHME version, renamed to 'Homicide' via column_name_mapping
    # 'Poisoning': poison_gap,  # Removed - using DrugDisorder (IHME) instead
    'Suicide': self_harm_ihme_gap,  # Using IHME version, renamed to 'Suicide' via column_name_mapping
    # 'MaternalMortality': maternal_gap,  # Removed - positive coefficient is suspect (higher female mortality should close gap, not widen it). Likely capturing general healthcare quality with limited variation in rich countries.
    'Neoplasms': neoplasms_ihme_gap,
    'LiverDisease': liver_disease_ihme_gap,  # Using IHME version
}

# Create predictor_dfs by applying OECD filtering to indicator_datasets
# Note: We will exclude Country and Year columns before merging
# We keep Mid (midpoint) and Gap columns as predictors, plus Male/Female columns for counterfactual analysis
predictor_dfs = {name: get_oecd(df) for name, df in indicator_datasets.items()}

In [ ]:
# Summary table of year coverage for each indicator
year_summary_df = summarize_years(predictor_dfs)
year_summary_df

In [ ]:
for name, df in predictor_dfs.items():
    drop_cols = ['Country', 'Year']
    # Keep Male, Female, Mid, and Gap columns for counterfactual analysis
    # Only drop Country and Year columns
    if drop_cols:
        predictor_dfs[name] = df.drop(columns=drop_cols)

# Check shapes after dropping Country and Year
for name, predictor_df in predictor_dfs.items():
    print(name, predictor_df.shape)

In [ ]:
# Merge all predictors on index (Country codes)
for name, df in predictor_dfs.items():
    analysis_df = analysis_df.join(df, how='outer')

# Display shape and column names
analysis_df.shape

In [ ]:
analysis_df.head()

In [ ]:
# Create missing data report
missing_report = pd.DataFrame({
    'Indicator': analysis_df.columns,
    'Missing_Count': [analysis_df[col].isna().sum() for col in analysis_df.columns],
    'Missing_Pct': [analysis_df[col].isna().sum() / len(analysis_df) * 100 for col in analysis_df.columns],
    'Available_Count': [analysis_df[col].notna().sum() for col in analysis_df.columns]
}).sort_values('Missing_Count', ascending=False)

missing_report

In [ ]:
# Show which countries have complete data for all indicators
complete_cases = analysis_df.dropna()
complete_cases.shape[0], f"{complete_cases.shape[0] / len(analysis_df) * 100:.1f}% of countries have complete data"

### Create Final Analysis Dataset

In [ ]:
# Use complete-case analysis for primary model
analysis_complete = analysis_df.dropna()

# Document excluded countries
excluded_countries = set(analysis_df.index) - set(analysis_complete.index)
excluded_countries if excluded_countries else "No countries excluded - all OECD countries have complete data"

In [ ]:
# Separate target and predictors
# Create both target variables
target_hale = analysis_complete['HALE_gap']
target_le = analysis_complete['LifeExpectancy_gap']
# Keep all predictor columns including Male, Female, Mid, and Gap for counterfactual analysis
# Only drop the target variable columns
predictors = analysis_complete.drop(columns=[
    'HALE_gap', 'HALE_Years_Male', 'HALE_Years_Female',
    'LifeExpectancy_gap', 'LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female'
])

# Display final dataset info
pd.DataFrame({
    'Dataset': ['Complete Cases'],
    'Countries': [len(analysis_complete)],
    'Target_Variables': ['HALE_gap, LifeExpectancy_gap'],
    'Number_of_Predictors': [len(predictors.columns)],
    'Predictor_Names': [', '.join(predictors.columns)]
})

## Descriptive Statistics

### HALE Gender Gap 


In [ ]:
# Summary statistics for target variable (HALE gap)
target_hale.describe()

In [ ]:
# Distribution of HALE gap across OECD countries
plt.hist(target_hale, bins=15, color=AIBM_COLORS['crimson'], edgecolor='white')
decorate(xlabel='HALE Gap (Female - Male, years)', 
         ylabel='Number of Countries',
         title='Distribution of HALE Gender Gap Across OECD Countries')

In [ ]:
# Identify potential outliers using IQR method
Q1 = target_hale.quantile(0.25)
Q3 = target_hale.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = target_hale[(target_hale < lower_bound) | (target_hale > upper_bound)]
outliers_df = pd.DataFrame({
    'Country': outliers.index,
    'HALE_Gap': outliers.values
}).sort_values('HALE_Gap')

outliers_df if not outliers_df.empty else "No outliers detected using IQR method"

### Life Expectancy Gender Gap 


In [ ]:
# Summary statistics for target variable (Life Expectancy gap)
target_le.describe()

In [ ]:
# Distribution of Life Expectancy gap across OECD countries
plt.hist(target_le, bins=15, color=AIBM_COLORS['blue'], edgecolor='white')
decorate(xlabel='Life Expectancy Gap (Female - Male, years)', 
         ylabel='Number of Countries',
         title='Distribution of Life Expectancy Gender Gap Across OECD Countries')

In [ ]:
# Identify potential outliers using IQR method
Q1 = target_le.quantile(0.25)
Q3 = target_le.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = target_le[(target_le < lower_bound) | (target_le > upper_bound)]
outliers_df = pd.DataFrame({
    'Country': outliers.index,
    'LifeExpectancy_Gap': outliers.values
}).sort_values('LifeExpectancy_Gap')

outliers_df if not outliers_df.empty else "No outliers detected using IQR method"

### Comparison of HALE and Life Expectancy Gaps

In [ ]:
# Compare the two target variables
comparison_df = pd.DataFrame({
    'HALE_Gap': target_hale,
    'LifeExpectancy_Gap': target_le
})

comparison_df.describe()

In [ ]:
# Scatter plot comparing HALE gap vs Life Expectancy gap
plt.scatter(target_hale, target_le, color=AIBM_COLORS['crimson'], alpha=0.6)
decorate(xlabel='HALE Gap (Female - Male, years)', 
         ylabel='Life Expectancy Gap (Female - Male, years)',
         title='HALE Gap vs Life Expectancy Gap Across OECD Countries')

In [ ]:
# Correlation between the two target variables
correlation = target_hale.corr(target_le)
print(f"Correlation between HALE gap and Life Expectancy gap: {correlation:.3f}")

### Extreme Values and Country Rankings

In [ ]:
# Get list of indicators from indicator_datasets
indicator_names = list(indicator_datasets.keys())

# Create tables for each indicator showing top 5 and bottom 5 countries
indicator_extremes = {}

for indicator_name in indicator_names:
    # Get the dataframe directly from indicator_datasets
    df_gap = indicator_datasets[indicator_name]
    
    # Find Male and Female columns dynamically (they end with _Male or _Female)
    male_cols = [col for col in df_gap.columns if col.endswith('_Male')]
    female_cols = [col for col in df_gap.columns if col.endswith('_Female')]
    
    # Get the base name from the first male column (remove _Male suffix)
    if male_cols and female_cols:
        # Use the first matching pair
        male_col = male_cols[0]
        female_col = female_cols[0]
        # Verify they have the same base name
        base_name_male = male_col[:-5]  # Remove '_Male'
        base_name_female = female_col[:-7]  # Remove '_Female'
        if base_name_male == base_name_female:
            # Create summary for this indicator
            indicator_data = df_gap[[male_col, female_col]].copy()
            indicator_data['Country'] = df_gap['Country']
            
            # For male values
            male_sorted = indicator_data.sort_values(male_col, ascending=False)
            male_top5 = male_sorted.head(5)[['Country', male_col]].copy()
            male_bottom5 = male_sorted.tail(5)[['Country', male_col]].copy()
            
            # For female values
            female_sorted = indicator_data.sort_values(female_col, ascending=False)
            female_top5 = female_sorted.head(5)[['Country', female_col]].copy()
            female_bottom5 = female_sorted.tail(5)[['Country', female_col]].copy()
            
            indicator_extremes[indicator_name] = {
                'male_top5': male_top5,
                'male_bottom5': male_bottom5,
                'female_top5': female_top5,
                'female_bottom5': female_bottom5
            }

# Display results for a few key indicators
key_indicators = ['Alcohol', 'Homicide', 'Cardiovascular', 'Suicide']
for indicator in key_indicators:
    if indicator in indicator_extremes:
        print(f"\n{'='*60}")
        print(f"{indicator} - Extreme Values")
        print(f"{'='*60}")
        print(f"\nTop 5 Countries - Male Values:")
        display(indicator_extremes[indicator]['male_top5'])
        print(f"\nBottom 5 Countries - Male Values:")
        display(indicator_extremes[indicator]['male_bottom5'])
        print(f"\nTop 5 Countries - Female Values:")
        display(indicator_extremes[indicator]['female_top5'])
        print(f"\nBottom 5 Countries - Female Values:")
        display(indicator_extremes[indicator]['female_bottom5'])

#### For Gender Gaps: Largest and Smallest Gaps

In [ ]:
gap_extremes = {}

for indicator_name in indicator_names:
    # Get the dataframe directly from indicator_datasets
    df_gap = indicator_datasets[indicator_name]
    
    # Find columns dynamically
    gap_cols = [col for col in df_gap.columns if col.startswith('Gap_')]
    male_cols = [col for col in df_gap.columns if col.endswith('_Male')]
    female_cols = [col for col in df_gap.columns if col.endswith('_Female')]
    
    # Find matching columns (same base name)
    if gap_cols and male_cols and female_cols:
        # Get the base name from the gap column (remove 'Gap_' prefix)
        gap_col = gap_cols[0]
        base_name = gap_col[4:]  # Remove 'Gap_' prefix
        
        # Find matching male and female columns
        male_col = f'{base_name}_Male'
        female_col = f'{base_name}_Female'
        
        if male_col in df_gap.columns and female_col in df_gap.columns:
            # Create summary with country names
            gap_data = df_gap[[male_col, female_col, gap_col]].copy()
            gap_data['Country'] = df_gap['Country']
            
            # Sort by gap (largest positive gaps first)
            # Note: Gap is computed as Male - Female, so positive means males have higher rates
            gap_sorted = gap_data.sort_values(gap_col, ascending=False)
            
            gap_top5 = gap_sorted.head(5).copy()
            gap_bottom5 = gap_sorted.tail(5).copy()
            
            gap_extremes[indicator_name] = {
                'top5': gap_top5,
                'bottom5': gap_bottom5
            }

# Display results for key indicators
for indicator in key_indicators:
    if indicator in gap_extremes:
        print(f"\n{'='*60}")
        print(f"{indicator} - Gender Gap Extremes (Male - Female)")
        print(f"{'='*60}")
        gap_info = gap_extremes[indicator]
        # Get column names dynamically from the DataFrame
        top5_df = gap_info['top5']
        # Find Male, Female, and Gap columns
        male_col = [col for col in top5_df.columns if col.endswith('_Male')][0] if any(col.endswith('_Male') for col in top5_df.columns) else None
        female_col = [col for col in top5_df.columns if col.endswith('_Female')][0] if any(col.endswith('_Female') for col in top5_df.columns) else None
        gap_col = [col for col in top5_df.columns if col.startswith('Gap_')][0] if any(col.startswith('Gap_') for col in top5_df.columns) else None
        
        cols_to_show = ['Country']
        if male_col:
            cols_to_show.append(male_col)
        if female_col:
            cols_to_show.append(female_col)
        if gap_col:
            cols_to_show.append(gap_col)
        
        print(f"\nTop 5 Countries - Largest Gaps (Male > Female):")
        display(gap_info['top5'][cols_to_show])
        print(f"\nBottom 5 Countries - Smallest Gaps (Female > Male):")
        display(gap_info['bottom5'][cols_to_show])

#### For HALE Gap: All Countries Ranked

In [ ]:
# Create ranked table of all countries by HALE gap
hale_ranked = analysis_complete[['HALE_Years_Male', 'HALE_Years_Female', 'HALE_gap']].copy()
hale_ranked = hale_ranked.sort_values('HALE_gap', ascending=False).reset_index()
hale_ranked['Rank'] = range(1, len(hale_ranked) + 1)
# Rename: Country (code) -> Code, add Country (name)
hale_ranked = hale_ranked.rename(columns={'Country': 'Code'})
hale_ranked['Country'] = hale_ranked['Code'].map(code_to_who_country)

# Create output version without Rank column (Rank kept internally for comparison table)
hale_ranked_output = hale_ranked[['Country', 'HALE_Years_Male', 
                                   'HALE_Years_Female', 'HALE_gap']].copy()

print("All Countries Ranked by HALE Gap (Female - Male)")
print("="*80)
hale_ranked_output

In [ ]:
# Write HALE gap by country table to HTML (without Rank column)
write_html_table(hale_ranked_output, "jb/tables/hale_gap_by_country.html")

In [ ]:
# Summary statistics for HALE gap
print("\nHALE Gap Summary Statistics:")
print("="*50)
print(f"Mean HALE Gap: {hale_ranked['HALE_gap'].mean():.2f} years")
print(f"Median HALE Gap: {hale_ranked['HALE_gap'].median():.2f} years")
print(f"Standard Deviation: {hale_ranked['HALE_gap'].std():.2f} years")
print(f"Range: {hale_ranked['HALE_gap'].min():.2f} to {hale_ranked['HALE_gap'].max():.2f} years")
print(f"\nCountries with Largest HALE Gap (Top 5):")
hale_ranked_output.head(5)
print(f"\nCountries with Smallest HALE Gap (Bottom 5):")
hale_ranked_output.tail(5)

#### For Life Expectancy Gap: All Countries Ranked

In [ ]:
# Create ranked table of all countries by Life Expectancy gap
le_ranked = analysis_complete[['LifeExpectancy_Years_Male', 'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']].copy()
le_ranked = le_ranked.sort_values('LifeExpectancy_gap', ascending=False).reset_index()
le_ranked['Rank'] = range(1, len(le_ranked) + 1)
# Rename: Country (code) -> Code, add Country (name)
le_ranked = le_ranked.rename(columns={'Country': 'Code'})
le_ranked['Country'] = le_ranked['Code'].map(code_to_who_country)

# Create output version without Rank column (Rank kept internally for comparison table)
le_ranked_output = le_ranked[['Country', 'LifeExpectancy_Years_Male', 
                              'LifeExpectancy_Years_Female', 'LifeExpectancy_gap']].copy()

print("All Countries Ranked by Life Expectancy Gap (Female - Male)")
print("="*80)
le_ranked_output

In [ ]:
# Write Life Expectancy gap by country table to HTML (without Rank column)
write_html_table(le_ranked_output, "jb/tables/le_gap_by_country.html")

In [ ]:
# Summary statistics for Life Expectancy gap
print("\nLife Expectancy Gap Summary Statistics:")
print("="*50)
print(f"Mean Life Expectancy Gap: {le_ranked['LifeExpectancy_gap'].mean():.2f} years")
print(f"Median Life Expectancy Gap: {le_ranked['LifeExpectancy_gap'].median():.2f} years")
print(f"Standard Deviation: {le_ranked['LifeExpectancy_gap'].std():.2f} years")
print(f"Range: {le_ranked['LifeExpectancy_gap'].min():.2f} to {le_ranked['LifeExpectancy_gap'].max():.2f} years")
print(f"\nCountries with Largest Life Expectancy Gap (Top 5):")
le_ranked_output.head(5)
print(f"\nCountries with Smallest Life Expectancy Gap (Bottom 5):")
le_ranked_output.tail(5)

### Summary Statistics by Indicator

In [ ]:
# Create summary table with rate and gap statistics for each indicator
def compute_indicator_stats(indicator_name, df_gap):
    """Compute statistics for a single indicator."""
    # Get OECD countries only for consistency
    df_oecd = get_oecd(df_gap)
    
    # Find Mid and Gap columns dynamically
    mid_cols = [col for col in df_gap.columns if col.startswith('Mid_')]
    gap_cols = [col for col in df_gap.columns if col.startswith('Gap_')]
    
    # Handle normal case: indicators with both Mid and Gap columns
    if mid_cols and gap_cols:
        mid_col = mid_cols[0]
        gap_col = gap_cols[0]
        
        # Extract midpoint and gap values
        midpoints = df_oecd[mid_col].dropna()
        gaps = df_oecd[gap_col].dropna()
        
        if len(midpoints) > 0 and len(gaps) > 0:
            return {
                'Indicator': indicator_name,
                'Median Rate': midpoints.median(),
                'Min Rate': midpoints.min(),
                'Max Rate': midpoints.max(),
                'Median Gap': gaps.median(),
                'Min Gap': gaps.min(),
                'Max Gap': gaps.max()
            }
    
    # Handle special case: MaternalMortality (female-only, no Gap/Mid columns)
    # Use the Female value as both midpoint and gap
    elif indicator_name == 'MaternalMortality':
        female_cols = [col for col in df_gap.columns if col.endswith('_Female')]
        if female_cols:
            female_col = female_cols[0]
            values = df_oecd[female_col].dropna()
            
            if len(values) > 0:
                return {
                    'Indicator': indicator_name,
                    'Median Rate': values.median(),
                    'Min Rate': values.min(),
                    'Max Rate': values.max(),
                    'Median Gap': -values.median(),
                    'Min Gap': -values.min(),
                    'Max Gap': -values.max()
                }
    
    return None

In [ ]:
# Process predictor indicators
predictor_summary = []
for indicator_name, df_gap in indicator_datasets.items():
    stats = compute_indicator_stats(indicator_name, df_gap)
    if stats:
        predictor_summary.append(stats)

# Process target variables
target_summary = []
target_indicators = {
    'HALE': hale_gap,
    'Life Expectancy': le_gap
}
for indicator_name, df_gap in target_indicators.items():
    stats = compute_indicator_stats(indicator_name, df_gap)
    if stats:
        target_summary.append(stats)

In [ ]:
# Create DataFrames (keep Indicator as a column, not index)
predictor_df = pd.DataFrame(predictor_summary)
target_df = pd.DataFrame(target_summary)

# Split predictor table into rates and gaps
predictor_rates = predictor_df[['Indicator', 'Median Rate', 'Min Rate', 'Max Rate']].copy()
predictor_gaps = predictor_df[['Indicator', 'Median Gap', 'Min Gap', 'Max Gap']].copy()

# Calculate correlations with target variables
# For rates: use Mid_ columns, for gaps: use Gap_ columns
# Special case: MaternalMortality uses Female column for both
predictor_rates['Corr HALE'] = np.nan
predictor_rates['Corr LE'] = np.nan
predictor_gaps['Corr HALE'] = np.nan
predictor_gaps['Corr LE'] = np.nan

for idx, indicator in enumerate(predictor_rates['Indicator']):
    if indicator == 'MaternalMortality':
        # MaternalMortality: use Female column for both rates and gaps (if present)
        female_col = 'MaternalMortality_Female'
        if female_col in predictors.columns:
            predictor_rates.loc[idx, 'Corr HALE'] = predictors[female_col].corr(target_hale)
            predictor_rates.loc[idx, 'Corr LE'] = predictors[female_col].corr(target_le)
            predictor_gaps.loc[idx, 'Corr HALE'] = predictors[female_col].corr(target_hale)
            predictor_gaps.loc[idx, 'Corr LE'] = predictors[female_col].corr(target_le)
        else:
            # MaternalMortality not in predictors, skip correlation calculation
            predictor_rates.loc[idx, 'Corr HALE'] = np.nan
            predictor_rates.loc[idx, 'Corr LE'] = np.nan
            predictor_gaps.loc[idx, 'Corr HALE'] = np.nan
            predictor_gaps.loc[idx, 'Corr LE'] = np.nan
    else:
        # Find the corresponding Mid_ and Gap_ columns in predictors DataFrame
        mid_col = f'Mid_{indicator}'
        gap_col = f'Gap_{indicator}'
        
        # Calculate correlations for rates (using Mid_ columns)
        if mid_col in predictors.columns:
            predictor_rates.loc[idx, 'Corr HALE'] = predictors[mid_col].corr(target_hale)
            predictor_rates.loc[idx, 'Corr LE'] = predictors[mid_col].corr(target_le)
        else:
            predictor_rates.loc[idx, 'Corr HALE'] = np.nan
            predictor_rates.loc[idx, 'Corr LE'] = np.nan
        
        # Calculate correlations for gaps (using Gap_ columns)
        if gap_col in predictors.columns:
            predictor_gaps.loc[idx, 'Corr HALE'] = predictors[gap_col].corr(target_hale)
            predictor_gaps.loc[idx, 'Corr LE'] = predictors[gap_col].corr(target_le)
        else:
            predictor_gaps.loc[idx, 'Corr HALE'] = np.nan
            predictor_gaps.loc[idx, 'Corr LE'] = np.nan

# Split target table into rates and gaps, reverse gap signs, and adjust column names
target_rates = target_df[['Indicator', 'Median Rate', 'Min Rate', 'Max Rate']].copy()
target_rates.columns = ['Indicator', 'Median', 'Min', 'Max']
target_gaps = target_df[['Indicator', 'Median Gap', 'Min Gap', 'Max Gap']].copy()
# Reverse sign of gaps (from Male - Female to Female - Male)
target_gaps['Median Gap'] = -target_gaps['Median Gap']
target_gaps['Min Gap'] = -target_gaps['Min Gap']
target_gaps['Max Gap'] = -target_gaps['Max Gap']
# After negation, min and max are swapped, so swap the column values
target_gaps['Min Gap'], target_gaps['Max Gap'] = target_gaps['Max Gap'], target_gaps['Min Gap']
target_gaps.columns = ['Indicator', 'Median Gap', 'Min Gap', 'Max Gap']

In [ ]:
print("Predictor Indicators - Rates:")
predictor_rates = predictor_rates.sort_values(by='Median Rate', ascending=False).reset_index(drop=True)
predictor_rates

In [ ]:
print("Predictor Indicators - Gaps:")
predictor_gaps = predictor_gaps.sort_values(by='Median Gap', ascending=False).reset_index(drop=True)
predictor_gaps

In [ ]:
print("Target Variables - Rates:")
target_rates.sort_values(by='Median', ascending=False).reset_index(drop=True)

In [ ]:
print("Target Variables - Gaps:")
target_gaps

In [ ]:
write_html_table(predictor_rates, "jb/tables/predictor_rates.html")
write_html_table(predictor_gaps,  "jb/tables/predictor_gaps.html")

write_html_table(target_rates,    "jb/tables/target_rates.html")
write_html_table(target_gaps,     "jb/tables/target_gaps.html")

In [ ]:
# Create table showing correlation between Rate (Mid) and Gap for each indicator
rate_gap_correlations = []

for indicator in predictor_rates['Indicator']:
    # Skip MaternalMortality - it doesn't have a Gap column (female-only)
    if indicator == 'MaternalMortality':
        continue
    
    # Find the corresponding Mid_ and Gap_ columns in predictors DataFrame
    rate_col = f'Mid_{indicator}'
    gap_col = f'Gap_{indicator}'
    
    # Calculate correlation between rate and gap
    if rate_col in predictors.columns and gap_col in predictors.columns:
        corr = predictors[rate_col].corr(predictors[gap_col])
        rate_gap_correlations.append({
            'Indicator': indicator,
            'Correlation': corr
        })

# Create DataFrame and format (keep Indicator as a column)
rate_gap_corr_df = pd.DataFrame(rate_gap_correlations)

# Sort by correlation (descending)
rate_gap_corr_df = rate_gap_corr_df.sort_values('Correlation', ascending=False).reset_index(drop=True)

In [ ]:
write_html_table(rate_gap_corr_df, "jb/tables/rate_gap_correlation.html")

In [ ]:
# Create table showing top 10 correlations between rates (Mid columns)
# Get all Mid_ columns
mid_cols = [col for col in predictors.columns if col.startswith('Mid_')]

# Calculate correlation matrix for rates
rates_corr_matrix = predictors[mid_cols].corr()

# Extract upper triangle (excluding diagonal) and convert to list of pairs
rates_corr_pairs = []
for i in range(len(rates_corr_matrix.columns)):
    for j in range(i+1, len(rates_corr_matrix.columns)):
        indicator1 = rates_corr_matrix.columns[i].replace('Mid_', '')
        indicator2 = rates_corr_matrix.columns[j].replace('Mid_', '')
        corr_val = rates_corr_matrix.iloc[i, j]
        rates_corr_pairs.append({
            'Rate 1': indicator1,
            'Rate 2': indicator2,
            'Correlation': corr_val
        })

# Create DataFrame and sort by absolute correlation
rates_corr_df = pd.DataFrame(rates_corr_pairs)
rates_corr_df['Abs Correlation'] = rates_corr_df['Correlation'].abs()
rates_corr_df = rates_corr_df.sort_values('Abs Correlation', ascending=False).head(10)

# Drop the absolute value column
rates_corr_df = rates_corr_df[['Rate 1', 'Rate 2', 'Correlation']].reset_index(drop=True)

In [ ]:
write_html_table(rates_corr_df,     "jb/tables/rate_rate_correlation_top10.html")

In [ ]:
# Create table showing top 10 correlations between gaps (Gap columns)
# Get all Gap_ columns (excluding MaternalMortality which doesn't have a Gap column)
gap_cols = [col for col in predictors.columns if col.startswith('Gap_')]

# Calculate correlation matrix for gaps
gaps_corr_matrix = predictors[gap_cols].corr()

# Extract upper triangle (excluding diagonal) and convert to list of pairs
gaps_corr_pairs = []
for i in range(len(gaps_corr_matrix.columns)):
    for j in range(i+1, len(gaps_corr_matrix.columns)):
        indicator1 = gaps_corr_matrix.columns[i].replace('Gap_', '')
        indicator2 = gaps_corr_matrix.columns[j].replace('Gap_', '')
        corr_val = gaps_corr_matrix.iloc[i, j]
        gaps_corr_pairs.append({
            'Gap 1': indicator1,
            'Gap 2': indicator2,
            'Correlation': corr_val
        })

# Create DataFrame and sort by absolute correlation
gaps_corr_df = pd.DataFrame(gaps_corr_pairs)
gaps_corr_df['Abs Correlation'] = gaps_corr_df['Correlation'].abs()
gaps_corr_df = gaps_corr_df.sort_values('Abs Correlation', ascending=False).head(10)

# Drop the absolute value column
gaps_corr_df = gaps_corr_df[['Gap 1', 'Gap 2', 'Correlation']].reset_index(drop=True)

In [ ]:
write_html_table(gaps_corr_df,     "jb/tables/gap_gap_correlation_top10.html")

### Save Data for Future Use

In [ ]:
# Save to HDF5 file
hdf_file = '../data/hale_analysis_data.h5'
with pd.HDFStore(hdf_file, mode='w') as store:
    store['predictors'] = predictors
    store['target_hale'] = target_hale
    store['target_le'] = target_le

In [ ]:
predictors.columns

In [ ]:
from utils import beep

beep()